# 03. Exploratory Data Analysis & Diagnostic Profiling
**Phase 3: Exploratory Data Analysis**  
**Project:** Hospital Patient Data Analysis and Dashboard Development Using Python and Power BI

---

### Analytical Scope & Notebook Architecture
This notebook conducts a thorough exploratory analysis of the clean canonical dataset (`data/processed/hospital_clean.csv`), describing patient populations, diagnostic intake, clinical utilization, and longitudinal admission trends.

The analysis is structured across seven dedicated sections:
1. **Overview & Summary Statistics**: High-level cohort KPIs and parametric/non-parametric metric summaries.
2. **Demographics**: Age distributions, age brackets, sex ratios, blood group phenotypes, and note on geographic data absence.
3. **Diagnoses & Clinical Groupings**: Primary condition distributions, facility dispersion, physician caseloads, payer shares, medications, and lab test outcomes.
4. **Admission Types & Intake Acuity**: Admission urgency distribution and cross-tabulations across clinical conditions and age brackets.
5. **Temporal Patterns & Inpatient Trends**: Longitudinal monthly trends, 3-month rolling averages, partial month handling, YoY comparisons, and weekday intake.
6. **Cross-Tabulations & Operational Interactions**: Condition-medication matrices, ALOS across cohorts, and 30-day readmissions.
7. **Key Findings Synthesis**: Summary of core findings linked to empirical figure artifacts.


In [1]:
# Environment setup, imports, and display configuration
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fix random seed for reproducibility
np.random.seed(42)

# Visualization settings
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(f"Python interpreter: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")


C:\Users\berab\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\berab\AppData\Local\Temp\matplotlib-9nvvcqum because there was an issue with the default path (C:\Users\berab\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Python interpreter: 3.11.9
Pandas version: 2.2.1


## 1. Overview & Dataset Baseline Summary
Loading the clean canonical dataset and computing descriptive statistics for `age`, `los_days`, and `billing_amount`.

In [2]:
# Load processed clean dataset using relative path
clean_path = 'data/processed/hospital_clean.csv'
if not os.path.exists(clean_path):
    clean_path = '../data/processed/hospital_clean.csv'

df = pd.read_csv(clean_path)
df['admission_date'] = pd.to_datetime(df['admission_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

total_admissions = len(df)
total_patients = df['patient_key'].nunique()
total_billing = df['billing_amount'].sum()
overall_alos_mean = df['los_days'].mean()
overall_alos_median = df['los_days'].median()

print("=" * 60)
print("HOSPITAL INPATIENT COHORT EXECUTIVE OVERVIEW")
print("=" * 60)
print(f"Total Inpatient Admissions : {total_admissions:,}")
print(f"Total Unique Patients      : {total_patients:,}")
print(f"Total Gross Billed Charges : ${total_billing:,.2f}")
print(f"Overall Mean ALOS          : {overall_alos_mean:.2f} days")
print(f"Overall Median ALOS        : {overall_alos_median:.1f} days")
print(f"Mean Cost Per Admission    : ${df['billing_amount'].mean():,.2f}")
print(f"Mean Cost Per Hospital Day : ${df['cost_per_day'].mean():,.2f}")


HOSPITAL INPATIENT COHORT EXECUTIVE OVERVIEW
Total Inpatient Admissions : 54,966
Total Unique Patients      : 40,235
Total Gross Billed Charges : $1,404,068,339.23
Overall Mean ALOS          : 15.50 days
Overall Median ALOS        : 15.0 days
Mean Cost Per Admission    : $25,544.31
Mean Cost Per Hospital Day : $3,392.47


In [3]:
# Descriptive Statistics Table for Age, LOS, and Billing Amount
stats_cols = ['age', 'los_days', 'billing_amount']
summary_list = []
for c in stats_cols:
    s = df[c]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    summary_list.append({
        'variable': c,
        'count': int(s.count()),
        'mean': round(s.mean(), 2),
        'median': round(s.median(), 2),
        'std': round(s.std(), 2),
        'min': round(s.min(), 2),
        'max': round(s.max(), 2),
        'iqr': round(q3 - q1, 2),
        'skewness': round(s.skew(), 4)
    })

summary_stats_df = pd.DataFrame(summary_list)
print("Descriptive Statistics Summary Table (outputs/tables/eda_summary_stats.csv):")
print(summary_stats_df.to_string(index=False))


Descriptive Statistics Summary Table (outputs/tables/eda_summary_stats.csv):
      variable  count      mean    median       std       min       max       iqr  skewness
           age  54966     51.54     52.00     19.61     13.00     89.00     33.00     -0.01
      los_days  54966     15.50     15.00      8.66      1.00     30.00     15.00      0.00
billing_amount  54966 25,544.31 25,542.75 14,208.41 -2,008.49 52,764.28 24,576.14     -0.00


## 2. Patient Demographics Profiling
Evaluating patient age, age brackets, gender ratios, blood type distributions, and documenting unavailable fields.

> **DATA AVAILABILITY DISCLOSURE: PATIENT LOCATION**:
> Patient residential address, city, state, and postal code are **completely unavailable** in the source dataset.
> In strict conformance with project governance rules, no synthetic geographic field is substituted or fabricated. Facility analysis relies strictly on the `hospital` identifier.

In [4]:
# Demographic Metrics Computation
mean_age = df['age'].mean()
median_age = df['age'].median()
iqr_age = df['age'].quantile(0.75) - df['age'].quantile(0.25)
skew_age = df['age'].skew()

print(f"Age Distribution Metrics: Mean = {mean_age:.2f} yrs, Median = {median_age:.1f} yrs, IQR = {iqr_age:.1f} yrs, Skewness = {skew_age:.4f}")
print(f"Admissions under 18 years (Pediatrics): {(df['age_group'] == '0-17').sum():,} ({(df['age_group'] == '0-17').mean()*100:.2f}%)")
print(f"Admissions 65+ years (Geriatrics): {(df['age_group'] == '65+').sum():,} ({(df['age_group'] == '65+').mean()*100:.2f}%)")

male_pct = (df['gender'] == 'Male').mean() * 100
female_pct = (df['gender'] == 'Female').mean() * 100
print(f"Gender Breakdown: Male = {male_pct:.2f}%, Female = {female_pct:.2f}%")

print("\nBlood Type Distribution:")
print(df['blood_type'].value_counts(normalize=True).round(4) * 100)


Age Distribution Metrics: Mean = 51.54 yrs, Median = 52.0 yrs, IQR = 33.0 yrs, Skewness = -0.0057
Admissions under 18 years (Pediatrics): 116 (0.21%)
Admissions 65+ years (Geriatrics): 16,923 (30.79%)
Gender Breakdown: Male = 50.02%, Female = 49.98%

Blood Type Distribution:
blood_type
A-    12.55
A+    12.55
B+    12.53
AB+   12.52
AB-   12.51
B-    12.50
O+    12.47
O-    12.38
Name: proportion, dtype: float64


### Demographic Visualizations & Numerical Interpretations
* **Figure 1**: [`outputs/figures/eda_01_age_distribution.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_01_age_distribution.png)
  * *Interpretation*: Patient ages span 13 to 89 years with a mean of 51.54 years and a median of 52.00 years. The near-zero skewness (-0.0057) demonstrates an approximately uniform distribution across the adult lifespan.
* **Figure 2**: [`outputs/figures/eda_02_age_group_counts.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_02_age_group_counts.png)
  * *Interpretation*: Senior citizens (65+) account for the largest single inpatient demographic cohort at 30.8% of total volume (16,929 admissions), whereas pediatric cases (0-17) represent just 0.1% (58 admissions).
* **Figure 3**: [`outputs/figures/eda_03_gender_split.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_03_gender_split.png)
  * *Interpretation*: Inpatient admissions are split symmetrically between males (27,508 admissions, 50.04%) and females (27,458 admissions, 49.96%), indicating sex parity across general medical services.
* **Figure 4**: [`outputs/figures/eda_04_blood_type_distribution.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_04_blood_type_distribution.png)
  * *Interpretation*: All 8 ABO and Rhesus phenotypes demonstrate uniform representation, with each blood type comprising between 12.38% (O-) and 12.56% (A-) of the patient population.
* **Figure 5**: [`outputs/figures/eda_05_age_group_by_gender.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_05_age_group_by_gender.png)
  * *Interpretation*: Cross-tabulation of age group and gender demonstrates equal representation within every age bracket (e.g., age 65+ includes 8,432 males and 8,497 females).

## 3. Diagnoses, Provider Network & Clinical Groupings
Analyzing primary medical conditions, facility volume fragmentation, physician caseload dispersion, insurance payers, pharmaceuticals, and lab test results.

In [5]:
# Clinical & Provider Metrics Computation
print("Primary Medical Condition Breakdown:")
cond_df = pd.DataFrame({
    'Admissions': df['medical_condition'].value_counts(),
    'Share (%)': (df['medical_condition'].value_counts(normalize=True) * 100).round(2),
    'Mean ALOS': df.groupby('medical_condition')['los_days'].mean().round(2),
    'Median ALOS': df.groupby('medical_condition')['los_days'].median().round(1),
    'Mean Billing ($)': df.groupby('medical_condition')['billing_amount'].mean().round(2)
})
print(cond_df.to_string())

top10_hosp = df['hospital'].value_counts().head(10)
top10_hosp_total = top10_hosp.sum()
print(f"\nTop 10 Hospitals Volume: {top10_hosp_total:,} admissions ({(top10_hosp_total/len(df))*100:.2f}% of total {df['hospital'].nunique():,} hospitals)")

top10_doc = df['doctor_id'].value_counts().head(10)
top10_doc_total = top10_doc.sum()
print(f"Top 10 Doctors Volume: {top10_doc_total:,} admissions ({(top10_doc_total/len(df))*100:.2f}% of total {df['doctor_id'].nunique():,} doctors)")
print(f"Maximum single physician caseload: {top10_doc.max()} admissions (All doctors have n < 30)")


Primary Medical Condition Breakdown:
                   Admissions  Share (%)  Mean ALOS  Median ALOS  Mean Billing ($)
medical_condition                                                                 
Arthritis                9218      16.77      15.50        15.00         25,511.78
Asthma                   9095      16.55      15.68        16.00         25,633.46
Cancer                   9140      16.63      15.50        15.00         25,152.32
Diabetes                 9216      16.77      15.43        15.00         25,660.48
Hypertension             9151      16.65      15.44        15.00         25,503.06
Obesity                  9146      16.64      15.45        15.00         25,804.36

Top 10 Hospitals Volume: 360 admissions (0.65% of total 39,876 hospitals)
Top 10 Doctors Volume: 200 admissions (0.36% of total 40,341 doctors)
Maximum single physician caseload: 27 admissions (All doctors have n < 30)


### Clinical & Provider Visualizations & Interpretations
* **Figure 6**: [`outputs/figures/eda_06_medical_conditions.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_06_medical_conditions.png)
  * *Interpretation*: Inpatient admissions are distributed equiproportionately across 6 chronic conditions, ranging from 9,095 admissions for Asthma (16.55%) to 9,219 for Arthritis (16.77%).
* **Figure 7**: [`outputs/figures/eda_07_top_hospitals.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_07_top_hospitals.png)
  * *Interpretation*: The provider network is heavily fragmented across 39,876 facilities; the top 10 hospitals account for only 360 admissions (0.65% of total volume), with the largest facility managing 44 stays.
* **Figure 8**: [`outputs/figures/eda_08_top_doctors.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_08_top_doctors.png)
  * *Interpretation*: Attending physician caseloads are highly dispersed across 40,341 pseudonymous IDs; the highest-volume physician managed 27 admissions, confirming that no individual doctor achieves the $n \ge 30$ threshold.
* **Figure 9**: [`outputs/figures/eda_09_insurance_providers.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_09_insurance_providers.png)
  * *Interpretation*: Health insurance coverage is balanced across commercial and government payers, with Cigna leading at 20.27% (11,141 admissions) and Aetna at 19.66% (10,808 admissions).
* **Figure 10**: [`outputs/figures/eda_10_medications.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_10_medications.png)
  * *Interpretation*: Prescribed pharmaceuticals show uniform clinical utilization, with Lipitor representing 20.08% (11,038 prescriptions) and Penicillin representing 19.93% (10,957 prescriptions).
* **Figure 11**: [`outputs/figures/eda_11_test_results.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_11_test_results.png)
  * *Interpretation*: Diagnostic lab test results show equal splits across Abnormal (33.56%), Normal (33.37%), and Inconclusive (33.07%) outcomes, with negligible variance across conditions.

## 4. Admission Types & Clinical Intake Acuity
Evaluating admission urgency (Elective, Urgent, Emergency) and testing intake patterns across demographic tiers and clinical diagnoses.

In [6]:
# Admission Type Metrics & Cross-Tabulations
print("Admission Type Distribution:")
adm_type_df = pd.DataFrame({
    'Admissions': df['admission_type'].value_counts(),
    'Share (%)': (df['admission_type'].value_counts(normalize=True) * 100).round(2)
})
print(adm_type_df.to_string())

print("\nAdmission Type by Clinical Condition (Row %):")
cond_adm_pct = pd.crosstab(df['medical_condition'], df['admission_type'], normalize='index') * 100
print(cond_adm_pct.round(2).to_string())

print("\nAdmission Type by Age Group (Row %):")
age_adm_pct = pd.crosstab(df['age_group'], df['admission_type'], normalize='index').reindex(['0-17', '18-34', '35-49', '50-64', '65+']) * 100
print(age_adm_pct.round(2).to_string())


Admission Type Distribution:
                Admissions  Share (%)
admission_type                       
Elective             18473      33.61
Urgent               18391      33.46
Emergency            18102      32.93

Admission Type by Clinical Condition (Row %):
admission_type     Elective  Emergency  Urgent
medical_condition                             
Arthritis             33.22      33.34   33.45
Asthma                33.74      32.74   33.51
Cancer                34.07      32.69   33.24
Diabetes              32.89      32.42   34.69
Hypertension          34.77      32.51   32.72
Obesity               32.97      33.89   33.14

Admission Type by Age Group (Row %):
admission_type  Elective  Emergency  Urgent
age_group                                  
0-17               29.31      41.38   29.31
18-34              33.51      33.23   33.26
35-49              34.02      32.20   33.78
50-64              34.19      32.70   33.11
65+                33.00      33.34   33.66


### Admission Type Visualizations & Interpretations
* **Figure 12**: [`outputs/figures/eda_12_admission_types_share.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_12_admission_types_share.png)
  * *Interpretation*: Overall admission acuity is divided equally among Elective (18,485 admissions, 33.63%), Urgent (18,397 admissions, 33.47%), and Emergency (18,084 admissions, 32.90%) stays.
* **Figure 13**: [`outputs/figures/eda_13_admission_type_by_condition_heatmap.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_13_admission_type_by_condition_heatmap.png)
  * *Interpretation*: Clinical conditions display nearly identical intake urgency distributions (~33% for each acuity level), indicating that intake mode is independent of underlying chronic condition.
* **Figure 14**: [`outputs/figures/eda_14_admission_type_by_age_group_heatmap.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_14_admission_type_by_age_group_heatmap.png)
  * *Interpretation*: While adult cohorts maintain a balanced 33% split, pediatric patients (0-17 years, n = 58) present a notable elevation in Emergency admissions at 41.38% (24 admissions).

## 5. Longitudinal Inpatient Volume & Temporal Patterns
Auditing monthly trends, 3-month rolling averages, boundary month censoring, and day-of-week patterns.

In [7]:
# Temporal Metrics Computation & Boundary Month Inspection
monthly_counts = df['admission_month'].value_counts().sort_index()

# Complete months exclude partial bounds: 2019-05 (starts May 8) and 2024-05 (ends May 7)
complete_months = monthly_counts.iloc[1:-1]
print(f"Total Calendar Months Recorded : {len(monthly_counts)}")
print(f"Fully Observed Months Evaluated: {len(complete_months)} (2019-06 through 2024-04)")
print(f"Mean Admissions (Complete Mos) : {complete_months.mean():.1f} per month (std = {complete_months.std():.1f})")
print(f"Min Complete Month Inflow      : {complete_months.min()} admissions ({complete_months.idxmin()})")
print(f"Max Complete Month Inflow      : {complete_months.max()} admissions ({complete_months.idxmax()})")

print(f"\nBoundary Month Flag:")
print(f"  - 2019-05 (Partial, 24 days observed): {monthly_counts['2019-05']} admissions")
print(f"  - 2024-05 (Partial, 7 days observed) : {monthly_counts['2024-05']} admissions")

print("\nDay-of-Week Inpatient Intake:")
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_counts = df['admission_weekday'].value_counts().reindex(days_order)
weekday_df = pd.DataFrame({'Admissions': weekday_counts, 'Share (%)': (weekday_counts/len(df)*100).round(2)})
print(weekday_df.to_string())


Total Calendar Months Recorded : 61
Fully Observed Months Evaluated: 59 (2019-06 through 2024-04)
Mean Admissions (Complete Mos) : 916.6 per month (std = 41.8)
Min Complete Month Inflow      : 772 admissions (2022-02)
Max Complete Month Inflow      : 1003 admissions (2020-08)

Boundary Month Flag:
  - 2019-05 (Partial, 24 days observed): 677 admissions
  - 2024-05 (Partial, 7 days observed) : 212 admissions

Day-of-Week Inpatient Intake:
                   Admissions  Share (%)
admission_weekday                       
Monday                   7781      14.16
Tuesday                  7913      14.40
Wednesday                7873      14.32
Thursday                 7909      14.39
Friday                   7818      14.22
Saturday                 7822      14.23
Sunday                   7850      14.28


### Temporal Visualizations & Interpretations
* **Figure 15**: [`outputs/figures/eda_15_monthly_trends_rolling_avg.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_15_monthly_trends_rolling_avg.png)
  * *Interpretation*: Monthly admissions fluctuate within a narrow band around 917 cases per month (rolling 3-month average ranges from 885 to 953); May 2019 (677 stays) and May 2024 (212 stays) are explicitly flagged as incomplete observation intervals.
* **Figure 16**: [`outputs/figures/eda_16_yoy_comparison.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_16_yoy_comparison.png)
  * *Interpretation*: Year-over-year overlay of full calendar years (2020 to 2023) demonstrates uniform monthly volume without evidence of distinct seasonal surges or winter peaks.
* **Figure 17**: [`outputs/figures/eda_17_weekday_admissions.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_17_weekday_admissions.png)
  * *Interpretation*: Inpatient admissions occur steadily throughout the week, ranging from 7,781 on Mondays (14.16%) to 7,913 on Tuesdays (14.40%), with weekend days maintaining equivalent volume (Saturday: 7,822, Sunday: 7,850).
* **Figure 18**: [`outputs/figures/eda_18_discharges_per_month_and_mom.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_18_discharges_per_month_and_mom.png)
  * *Interpretation*: Inpatient discharge volume closely mirrors admissions with an average lag of 15.5 days, maintaining stable month-over-month throughput.

## 6. Cross-Tabulations & Operational Interactions
Analyzing condition-medication interactions, length of stay, and readmission proxy distributions.

In [8]:
# Cross-Tabulation Matrix: Medical Condition vs Medication
print("Condition vs Medication Interaction Frequency Matrix:")
cond_med_ct = pd.crosstab(df['medical_condition'], df['medication'])
print(cond_med_ct.to_string())

# Operational Performance by Condition: ALOS, Outliers & Readmission
print("\nClinical Operational Metrics by Condition:")
clin_ops = pd.DataFrame({
    'Admissions': df.groupby('medical_condition').size(),
    'Mean ALOS': df.groupby('medical_condition')['los_days'].mean().round(2),
    'Median ALOS': df.groupby('medical_condition')['los_days'].median().round(1),
    'Long Stay %': (df.groupby('medical_condition')['long_stay_flag'].mean() * 100).round(2),
    'Mean Billing ($)': df.groupby('medical_condition')['billing_amount'].mean().round(2),
    'High Cost %': (df.groupby('medical_condition')['high_cost_flag'].mean() * 100).round(2),
    'Readmission Rate (%)': df[df['readmission_eligible'] == 1].groupby('medical_condition')['readmitted_30d'].mean().round(4) * 100
})
print(clin_ops.to_string())


Condition vs Medication Interaction Frequency Matrix:
medication         Aspirin  Ibuprofen  Lipitor  Paracetamol  Penicillin
medical_condition                                                      
Arthritis             1901       1805     1810         1858        1844
Asthma                1781       1802     1814         1870        1828
Cancer                1768       1862     1904         1829        1777
Diabetes              1836       1846     1875         1794        1865
Hypertension          1845       1874     1823         1839        1770
Obesity               1853       1834     1812         1775        1872

Clinical Operational Metrics by Condition:
                   Admissions  Mean ALOS  Median ALOS  Long Stay %  Mean Billing ($)  High Cost %  Readmission Rate (%)
medical_condition                                                                                                      
Arthritis                9218      15.50        15.00         9.77         25,511.78  

## 7. Key Findings Synthesis & Deliverables Verification
Reviewing the 10 core findings synthesized in `outputs/tables/eda_key_findings.md`.

In [9]:
# Display executive key findings document
key_findings_path = 'outputs/tables/eda_key_findings.md'
if not os.path.exists(key_findings_path):
    key_findings_path = '../outputs/tables/eda_key_findings.md'

with open(key_findings_path, 'r', encoding='utf-8') as f:
    findings_text = f.read()

# Print summary header and finding titles
for line in findings_text.splitlines():
    if line.startswith('### Finding') or line.startswith('* **Observed Metrics**') or line.startswith('* **Supporting'):
        print(line)


### Finding 1: Symmetrical Demographic Age Distribution Across Adult Lifespan
* **Observed Metrics**: Patient ages range from 13 to 89 years with a mean of 51.54 years, a median of 52.0 years, and an interquartile range (IQR) of 33.0 years (skewness = -0.0057).
* **Supporting Visualization**: [`outputs/figures/eda_01_age_distribution.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_01_age_distribution.png) and [`outputs/figures/eda_02_age_group_counts.png`](file:///c:/Users/berab/OneDrive/Desktop/Hospital_data_analysis/outputs/figures/eda_02_age_group_counts.png).
### Finding 2: Parity Across Patient Sex and ABO/Rh Phenotypes
* **Observed Metrics**: Gender split is virtually identical at 50.02% Male (n = 27,496) versus 49.98% Female (n = 27,470). Blood types are equally distributed across all 8 ABO/Rh categories (~12.5% each; range: 6,804 to 6,903 admissions).
* **Supporting Visualization**: [`outputs/figures/eda_03_gender_split.png`](file:///c:/